# Overview Plots

In this notebook, you can view the distribution plots to find specific diaObjIds.  Then you can look into the auto_plots folder to see more information about the star and its auto-generated phase curve.

## Importing Libraries and Loading Data

In [1]:
#import libraries and set up tap service
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import astropy.units as u
from lsst.rsp import get_tap_service
import pandas as pd
import os
import sys
pd.set_option('display.max_columns', None)

from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

from lsst.daf.butler import Butler
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles
)

#setting up butler
butler = Butler('dp1', collections='LSSTComCam/DP1')
assert butler is not None

import seaborn as sns
my_pal = sns.color_palette("colorblind")

In [2]:
outdir = os.getcwd()
auto_path = os.path.join(outdir,"auto_plots")

if not os.path.isdir(auto_path):
    os.mkdir(auto_path)

In [3]:
outdir = os.getcwd()
extra_auto_path = os.path.join(outdir,"auto_plots/more_plots")

if not os.path.isdir(extra_auto_path):
    os.mkdir(extra_auto_path)

In [4]:
outdir = os.getcwd()
png_auto_path = os.path.join(outdir,"auto_plots/png_plots")

if not os.path.isdir(png_auto_path):
    os.mkdir(png_auto_path)

In [5]:
diaObj_df = pd.read_csv("data/Extra_diaObj_df.csv",index_col=0)
forcedDiaObj_df = pd.read_csv("data/diaForced_df.csv",index_col=0)

grouped_forcedDia = forcedDiaObj_df.groupby("diaObjectId")

In [6]:
hue_ord = list(diaObj_df["DR3_field_loc_center"].unique())
hue_ord.sort()

In [7]:
filter_names = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()
filter_linestyles = get_multiband_plot_linestyles()

In [8]:
#adding in center point of fields from 301_0_DP1
field_centers = {
    "47 Tuc": [6.02, -72.08],
    "SV 38 7": [37.86, 6.98],
    "Fornax": [40.00, -34.45],
    "ECDFS": [53.13, -28.10],
    "EDFS": [59.10, -48.73],
    "SV 95 -25": [95.00, -25.00],
    "Seagull": [106.23, -10.51],
}
field_names = list(field_centers.keys())

In [9]:
#load in the maglim for each object
u_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='u')
g_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='g')
r_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='r')
i_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='i')
z_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='z')
y_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='y')

## Making Interactive Plots with All Matches

In [10]:
"""
Need to update mag-mags with filtered removed if desired
"""

'\nNeed to update mag-mags with filtered removed if desired\n'

In [11]:
diaObj_df["diaObjectId_str"] = diaObj_df["diaObjectId"].astype(str)
fig = px.scatter(
    diaObj_df, 
    x="DR3_phot_g_mean_mag", 
    y="mean_r_mag",
    color="nr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="R Mag in Rubin vs G Mag in GAIA",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "in_Obj":"In Object Table",
        "DR3_phot_g_mean_mag": "Average G Mag from GAIA",
        "mean_r_mag": "Average R Mag from Rubin",
        "nr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(diaObj_df["mean_r_mag"].min(), diaObj_df["DR3_phot_g_mean_mag"].min()),
    max(diaObj_df["mean_r_mag"].max(), diaObj_df["DR3_phot_g_mean_mag"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(autorange="reversed", title_font=dict(size=16)),
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/mag_mag_interactive.html")
#fig.write_image(f"{png_auto_path}/mag_mag_interactive.png",scale=3)
fig.show()

In [12]:
fig = px.scatter(
    diaObj_df, 
    x="nSources", 
    y="mean_r_mag",
    color="nr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'], 
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Mean R Mag versus # of Epochs",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "in_Obj":"In Object Table",
        "nSources": "Number of Epochs Observed",
        "mean_r_mag": "Average R Mag from Rubin",
        "nr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


fig.update_layout(
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    xaxis_type="log",
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/mag_epoch_interactive.html")
#fig.write_image(f"{png_auto_path}/mag_epoch_interactive.png",scale=3)
fig.show()

In [13]:
fig = px.scatter(
    diaObj_df, 
    x="DR3_phot_g_mean_mag", 
    y="mean_r_mag",
    color="nr",
    symbol="in_Obj",
    symbol_sequence=['circle', 'x'], 
    hover_name="diaObjectId_str", 
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="R Mag in Rubin vs G Mag in GAIA",
    hover_data=["DR3_field_loc_center","DR3_best_classification"],
    labels={
        "DR3_field_loc_center":"DP1 Field",
        "DR3_phot_g_mean_mag": "Average G Mag from GAIA",
        "mean_r_mag": "Average R Mag from Rubin",
        "nr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(diaObj_df["mean_r_mag"].min(), diaObj_df["DR3_phot_g_mean_mag"].min()),
    max(diaObj_df["mean_r_mag"].max(), diaObj_df["DR3_phot_g_mean_mag"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(autorange="reversed", title_font=dict(size=16)),
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)

fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/obj_mag_mag_interactive.html")
#fig.write_image(f"{png_auto_path}/obj_mag_mag_interactive.png",scale=3)
fig.show()

In [14]:
fig = px.scatter(
    diaObj_df, 
    x="nSources", 
    y="mean_r_mag",
    color="nr",
    symbol="in_Obj",
    symbol_sequence=['circle', 'x'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Mean R Mag versus # of Epochs",
    hover_data=["DR3_field_loc_center","DR3_best_classification"],
    labels={
        "DR3_field_loc_center":"DP1 Field",
        "nSources": "Number of Epochs Observed",
        "mean_r_mag": "Average R Mag from Rubin",
        "nr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


fig.update_layout(
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    xaxis_type="log",
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/obj_mag_epoch_interactive.html")
#fig.write_image(f"{png_auto_path}/obj_mag_epoch_interactive.png",scale=3)
fig.show()

## Making Interactive Plots after Filtering

Removed bad measurements from each group and filtered out stars with too few epochs

In [15]:
"""
Need to update mag-mags with filtered removed if desired
"""

'\nNeed to update mag-mags with filtered removed if desired\n'

In [16]:
mask = diaObj_df["nGoodSources"] >= 10

In [17]:
diaObj_df["diaObjectId_str"] = diaObj_df["diaObjectId"].astype(str)
fig = px.scatter(
    diaObj_df[mask], 
    x="DR3_phot_g_mean_mag", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="R Mag in Rubin vs G Mag in GAIA",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "in_Obj":"In Object Table",
        "DR3_phot_g_mean_mag": "Average G Mag from GAIA",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(diaObj_df["filt_mean_r_mag"].min(), diaObj_df["DR3_phot_g_mean_mag"].min()),
    max(diaObj_df["filt_mean_r_mag"].max(), diaObj_df["DR3_phot_g_mean_mag"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(autorange="reversed", title_font=dict(size=16)),
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/filt_mag_mag_interactive.html")
#fig.write_image(f"{png_auto_path}/filt_mag_mag_interactive.png",scale=3)
fig.show()

In [18]:
fig = px.scatter(
    diaObj_df[mask], 
    x="nSources", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'], 
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Mean R Mag versus # of Epochs",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "in_Obj":"In Object Table",
        "nSources": "Number of Epochs Observed",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


fig.update_layout(
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    xaxis_type="log",
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/filt_mag_epoch_interactive.html")
#fig.write_image(f"{png_auto_path}/filt_mag_epoch_interactive.png",scale=3)
fig.show()

In [19]:
fig = px.scatter(
    diaObj_df[mask], 
    x="DR3_phot_g_mean_mag", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="in_Obj",
    symbol_sequence=['circle', 'x'], 
    hover_name="diaObjectId_str", 
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="R Mag in Rubin vs G Mag in GAIA",
    hover_data=["DR3_field_loc_center","DR3_best_classification"],
    labels={
        "DR3_field_loc_center":"DP1 Field",
        "DR3_phot_g_mean_mag": "Average G Mag from GAIA",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(diaObj_df["filt_mean_r_mag"].min(), diaObj_df["DR3_phot_g_mean_mag"].min()),
    max(diaObj_df["filt_mean_r_mag"].max(), diaObj_df["DR3_phot_g_mean_mag"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(autorange="reversed", title_font=dict(size=16)),
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)

fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/filt_obj_mag_mag_interactive.html")
#fig.write_image(f"{png_auto_path}/filt_obj_mag_mag_interactive.png",scale=3)
fig.show()

In [20]:
fig = px.scatter(
    diaObj_df[mask], 
    x="nSources", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="in_Obj",
    symbol_sequence=['circle', 'x'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Mean R Mag versus # of Epochs",
    hover_data=["DR3_field_loc_center","DR3_best_classification"],
    labels={
        "DR3_field_loc_center":"DP1 Field",
        "nSources": "Number of Epochs Observed",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


fig.update_layout(
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    xaxis_type="log",
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/filt_obj_mag_epoch_interactive.html")
#fig.write_image(f"{png_auto_path}/filt_obj_mag_epoch_interactive.png",scale=3)
fig.show()

## Repeating with mask applied to auto_phase

In [21]:
#pull mask defined in dia_stats
mask = diaObj_df["comb_auto_filter"]

In [22]:
diaObj_df["diaObjectId_str"] = diaObj_df["diaObjectId"].astype(str)
fig = px.scatter(
    diaObj_df[mask], 
    x="DR3_phot_g_mean_mag", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="R Mag in Rubin vs G Mag in GAIA",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "in_Obj":"In Object Table",
        "DR3_phot_g_mean_mag": "Average G Mag from GAIA",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(diaObj_df["mean_r_mag"].min(), diaObj_df["DR3_phot_g_mean_mag"].min()),
    max(diaObj_df["mean_r_mag"].max(), diaObj_df["DR3_phot_g_mean_mag"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(autorange="reversed", title_font=dict(size=16)),
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{auto_path}/masked_mag_mag_interactive.html")
#fig.write_image(f"{png_auto_path}/masked_mag_mag_interactive.png",scale=3)
fig.show()

In [23]:
fig = px.scatter(
    diaObj_df[mask], 
    x="nSources", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'], 
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Mean R Mag versus # of Epochs",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "in_Obj":"In Object Table",
        "nSources": "Number of Epochs Observed",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


fig.update_layout(
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    xaxis_type="log",
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/masked_mag_epoch_interactive.html")
#fig.write_image(f"{png_auto_path}/masked_mag_epoch_interactive.png",scale=3)
fig.show()

In [24]:
fig = px.scatter(
    diaObj_df[mask], 
    x="DR3_phot_g_mean_mag", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="in_Obj",
    symbol_sequence=['circle', 'x'], 
    hover_name="diaObjectId_str", 
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="R Mag in Rubin vs G Mag in GAIA",
    hover_data=["DR3_field_loc_center","DR3_best_classification"],
    labels={
        "DR3_field_loc_center":"DP1 Field",
        "DR3_phot_g_mean_mag": "Average G Mag from GAIA",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(diaObj_df["filt_mean_r_mag"].min(), diaObj_df["DR3_phot_g_mean_mag"].min()),
    max(diaObj_df["filt_mean_r_mag"].max(), diaObj_df["DR3_phot_g_mean_mag"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(autorange="reversed", title_font=dict(size=16)),
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)

fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/masked_obj_mag_mag_interactive.html")
#fig.write_image(f"{png_auto_path}/masked_obj_mag_mag_interactive.png",scale=3)
fig.show()

In [25]:
fig = px.scatter(
    diaObj_df[mask], 
    x="nSources", 
    y="filt_mean_r_mag",
    color="nGoodr",
    symbol="in_Obj",
    symbol_sequence=['circle', 'x'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Mean R Mag versus # of Epochs",
    hover_data=["DR3_field_loc_center","DR3_best_classification"],
    labels={
        "DR3_field_loc_center":"DP1 Field",
        "nSources": "Number of Epochs Observed",
        "filt_mean_r_mag": "Average R Mag from Rubin",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


fig.update_layout(
    yaxis=dict(autorange="reversed", title_font=dict(size=16)),
    xaxis_type="log",
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{extra_auto_path}/masked_obj_mag_epoch_interactive.html")
#fig.write_image(f"{png_auto_path}/masked_obj_mag_epoch_interactive.png",scale=3)
fig.show()

## AutoLomb Interactive Overview 

In [26]:
assert False == True,"stopping code in case auto_lomb has not been run"

AssertionError: stopping code in case auto_lomb has not been run

In [ ]:
def get_p(row):

    if row["DR3_best_classification"] == "RRab":
        return row["DR3_pf"]
    else:
        return row["DR3_p1_o"]

lomb_diaObj_ps = pd.read_csv("data/lomb_diaObj_ps.csv")

lomb_dia_df = diaObj_df[diaObj_df["comb_auto_filter"]].merge(lomb_diaObj_ps,on="diaObjectId")
ps = lomb_dia_df.apply(get_p,axis=1)
lomb_dia_df["ps"] = ps

In [ ]:
diaObj_df["diaObjectId_str"] = diaObj_df["diaObjectId"].astype(str)
fig = px.scatter(
    lomb_dia_df, 
    x="ps", 
    y="lomb_p1",
    color="nGoodr",
    symbol="DR3_field_loc_center",
    symbol_sequence=['circle', 'x', 'square', 'cross', 'diamond'],
    hover_name="diaObjectId_str",  
    color_continuous_scale=px.colors.diverging.RdBu_r,
    title="Period from LombScargle vs in GAIA",
    hover_data=["in_Obj","DR3_best_classification"],
    labels={
        "ps":"Period from GAIA (d)",
        "lomb_p1":"Period from LombScargle (d)",
        "in_Obj":"In Object Table",
        "nGoodr": "nr",
        "DR3_best_classification":"RRL Class"
    }
)


line_range = [
    min(lomb_dia_df["lomb_p1"].min(), lomb_dia_df["lomb_p1"].min()),
    max(lomb_dia_df["lomb_p1"].max(), lomb_dia_df["lomb_p1"].max())
]

fig.add_trace(
    go.Scatter(
        x=line_range, 
        y=line_range, 
        mode="lines", 
        line=dict(color="black", dash="dot", width=1),
        name="1:1 Line",
        showlegend=True,
        hoverinfo='skip' 
    )
)


fig.update_layout(
    xaxis=dict(title_font=dict(size=16)),
    yaxis=dict(title_font=dict(size=16)),
    title_font=dict(size=20),
    template="plotly_white",
    legend_title_text="Field Location",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)" 
    )
)

id_list = diaObj_df["diaObjectId_str"].unique().tolist()

search_buttons = []

search_buttons.append(dict(
    args=[{"selectedpoints": [None]}],
    label="Clear Search",
    method="restyle"
))


for star_id in id_list:
    idx = diaObj_df.index[diaObj_df["diaObjectId_str"] == star_id].tolist()
    
    search_buttons.append(dict(
        args=[{"selectedpoints": [idx]}], # Highlights the specific point
        label=f"ID: {star_id}",
        method="restyle"
    ))

fig.update_layout(
    width=1000,  
    height=700,   
    margin=dict(l=50, r=50, t=80, b=50), 
    clickmode='event+select',
    updatemenus=[
        dict(
            buttons=search_buttons,
            direction="down",
            showactive=True,
            x=0.8, # Adjust position to not overlap your legend
            y=1.1,
            xanchor="left",
            yanchor="top",
            active=0
        )
    ]
)


fig.update_traces(marker=dict(line=dict(width=1, color='Black')))
fig.write_html(f"{auto_path}/masked_lomb_GAIA_interactive.html")
#fig.write_image(f"{png_auto_path}/masked_lomb_GAIA_interactive.png",scale=3)
fig.show()